In [1]:
import os
os.environ["OOPAO_BACKEND"] = "cuda"
os.environ["OOPAO_PRECISION"] = "32"

In [2]:
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from OOPAO.calibration.InteractionMatrix import InteractionMatrix

try:
    import tomllib
except ImportError:  # Python < 3.11
    import tomli as tomllib

from aott.config import CONFIG_DIR, LoadInstrument



     °          *      *      
 ▄██▄   ▄██▄  ▄███▄   ▄██▄ * ▄██▄ 
██* ██ ██  ██ ██  ██ ██  ██ ██  ██
██  ██ ██° ██ ██  ██ ██* ██ ██  ██
██  ██ ██  ██ ████▀  ██▄▄██ ██  ██
██* ██ ██  ██ ██     ██▀▀██ ██  ██
██  ██ ██  ██ ██ *   ██  ██ ██* ██
 ▀██▀   ▀██▀  ██   ° ██  ██  ▀██▀ 
      *         *             


OOPAO Warning: 
Significant changes were done to the OOPAO repository, the Telescope class is no longer the "master" class and the Source is now carrying the EM-field info.


In [3]:
# The filled-in example config files, so the simulation runs without editing config/:
# the instrument from example_instrument.toml, the output folder from the [output]
# section of windows_dev.toml (relative to the repo root)
config = LoadInstrument(CONFIG_DIR / "example_instrument.toml")

with open(CONFIG_DIR / "windows_dev.toml", "rb") as f:
    hdf5_dir = CONFIG_DIR.parent / tomllib.load(f)["output"]["hdf5_dir"]

In [4]:
# %%
plt.ion()
# number of subaperture for the WFS
n_subaperture = config['wfs']['n_lenslets_across']


# %%-----------------------     TELESCOPE   ----------------------------------
from OOPAO.Telescope import Telescope

# create the Telescope object
tel = Telescope(resolution           = 6*n_subaperture,                          # resolution of the telescope in [pix]
                diameter             = config['telescope']['diameter_m'],                                        # diameter in [m]        
                samplingTime         = 1/1000,                                   # Sampling time in [s] of the AO loop
                centralObstruction   = config['telescope']['obstruction_ratio'],                                      # Central obstruction in [%] of a diameter 
                display_optical_path = False,                                    # Flag to display optical path
                fov                  = 0)                                     # field of view in [arcsec]. If set to 0 (default) this speeds up the computation of the phase screens but is uncompatible with off-axis targets


#%% -----------------------     NGS   ----------------------------------
from OOPAO.Source import Source

# create the Natural Guide Star object
ngs = Source(optBand     = 'R',           # Optical band (see photometry.py)
             magnitude   = 5)

# combine the NGS to the telescope using '*'
ngs*tel

# create the Scientific Target object located at 10 arcsec from the  ngs
src = Source(optBand     = 'J2',           # Optical band (see photometry.py)
             magnitude   = 5)

# combine the SRC to the telescope using '*'
src*tel


#%% -----------------------     ATMOSPHERE   ----------------------------------
from OOPAO.Atmosphere import Atmosphere
           
# create the Atmosphere object
atm = Atmosphere(telescope     = tel,                               # Telescope                              
                 r0            = 0.05,                              # Fried Parameter [m]
                 L0            = 25,                                # Outer Scale [m]
                 fractionalR0  = [0.45 ,0.1  ,0.1  ,0.25  ,0.1   ], # Cn2 Profile
                 windSpeed     = [10   ,12   ,11   ,15    ,20    ], # Wind Speed in [m]
                 windDirection = [0    ,72   ,144  ,216   ,288   ], # Wind Direction in [degrees]
                 altitude      = [0    ,1000 ,5000 ,10000 ,12000 ]) # Altitude Layers in [m]


# initialize atmosphere with current Telescope
atm.initializeAtmosphere(tel)

# The phase screen can be updated using atm.update method (Temporal sampling given by tel.samplingTime)
atm.update()

#%% -----------------------     DEFORMABLE MIRROR   ----------------------------------
from OOPAO.DeformableMirror import DeformableMirror
from OOPAO.MisRegistration import MisRegistration

# mis-registrations object (rotation, shifts..)
misReg = MisRegistration()
misReg.shiftX = 0           # in [m]
misReg.shiftY = 0           # in [m]
misReg.rotationAngle = 0    # in [deg]


# specifying a given number of actuators along the diameter: 
nAct = config['dm']['actuators_in_diameter']
    
dm = DeformableMirror(telescope  = tel,                        # Telescope
                    nSubap       = nAct-1,                     # number of subaperture of the system considered (by default the DM has n_subaperture + 1 actuators to be in a Fried Geometry)
                    mechCoupling = 0.35,                       # Mechanical Coupling for the influence functions
                    misReg       = misReg,                     # Mis-registration associated 
                    coordinates  = None,                       # coordinates in [m]. Should be input as an array of size [n_actuators, 2] 
                    sign         = 1e-5,                       # Stroke
                    pitch        = tel.D/nAct)                 # inter actuator distance. Only used to compute the influence function coupling. The default is based on the n_subaperture value. 


#%% -----------------------     Pyramid WFS   ----------------------------------
from OOPAO.Pyramid import Pyramid

# make sure that the ngs is propagated to the wfs
ngs*tel

wfs = Pyramid(nSubap            = n_subaperture,                # number of subaperture = number of pixel accros the pupil diameter
              telescope         = tel,                          # telescope object
              lightRatio        = 0.5,                          # flux threshold to select valid sub-subaperture
              modulation        = config['wfs']['modulation'],                            # Tip tilt modulation radius
              binning           = 1,                            # binning factor (applied only on the )
              n_pix_separation  = 2,                            # number of pixel separating the different pupils
              n_pix_edge        = 1,                            # number of pixel on the edges of the pupils
              postProcessing    = 'fullFrame_incidence_flux')  # slopesMap_incidence_flux, fullFrame_incidence_flux (see documentation)



#%% Adjust the flux considering number of photons per subap.

n_photons_per_subap = 1000

surface_telescope = tel.pixelArea* tel.pixelSize*tel.pixelSize

if wfs.postProcessing[:10] == 'slopesMaps':    
    n_valid_subap = np.sum(wfs.validSignal)/2
else:
    n_valid_subap = np.sum(wfs.validSignal)/4

ngs.nPhoton = n_photons_per_subap / tel.samplingTime / (surface_telescope/n_valid_subap)       # nPhoton = # photons per s per m2

ngs*tel*wfs



#%% -----------------------     Modal Basis - Zernike  ----------------------------------
from OOPAO.Zernike import Zernike

#% ZERNIKE Polynomials
# create Zernike Object
Z = Zernike(tel,50)
# compute polynomials for given telescope
Z.computeZernike(tel)

# # mode to command matrix to project Zernike Polynomials on DM
M2C_zernike = np.linalg.pinv(np.squeeze(dm.modes[tel.pupilLogical,:]) * 2 * np.pi / ngs.wavelength)@Z.modes
C2Z = np.linalg.pinv(M2C_zernike)

#%% -----------------------     Modal Basis - KL Basis  ----------------------------------


from OOPAO.calibration.compute_KL_modal_basis import compute_KL_basis
# use the default definition of the KL modes with forced Tip and Tilt. For more complex KL modes, consider the use of the compute_KL_basis function. 
M2C_KL = compute_KL_basis(tel,
                          atm,
                          dm,
                          lim = 0) # inversion stability criterion

#%% -----------------------     Calibration: Interaction Matrix  ----------------------------------

# amplitude of the modes in m
stroke=1e-9
# zonal Interaction Matrix
M2C_zonal = np.eye(dm.nValidAct)

# modal Interaction Matrix for 300 modes
M2C_modal = M2C_KL[:,:300]

# swap to geometric WFS for the calibration
ngs**tel*wfs # make sure that the proper source is propagated to the WFS
# zonal interaction matrix
calib_modal = InteractionMatrix(ngs            = ngs,
                                atm            = atm,
                                tel            = tel,
                                dm             = dm,
                                wfs            = wfs,   
                                M2C            = M2C_modal, # M2C matrix used 
                                stroke         = stroke,    # stroke for the push/pull in M2C units
                                nMeasurements  = 12,        # number of simultaneous measurements
                                noise          = 'off',     # disable wfs.cam noise 
                                display        = True,      # display the time using tqdm
                                single_pass    = True)      # only push to compute the interaction matrix instead of push-pull



#%% Define instrument and WFS path detectors
from OOPAO.Detector import Detector
# instrument path
src_cam = Detector(tel.resolution*2,
                    readoutNoise    = config['science_camera']['ron'],  # readout of the detector in [e-/pixel]
                    QE              = 1,                   # quantum efficiency
                    psf_sampling    = config['science_camera']['sampling_at_calibration'])
src_cam.integrationTime = tel.samplingTime # exposure time for the PSF


# WFS path
ngs_cam = Detector(tel.resolution*2)
ngs_cam.psf_sampling = 4
ngs_cam.integrationTime = tel.samplingTime

ngs**tel*ngs_cam
ngs_psf_ref = ngs_cam.frame.copy()

src**tel*src_cam

src_psf_ref = src_cam.frame.copy()

#%%  Closed loop simulation
from OOPAO.tools.tools import strehlMeter


------------ Telescope -------------
Diameter [m]             |   3.00   
Resolution [px]          |   120    
Pixel size [m]           |   0.03   
Surface [m²]             |   6.43   
Central obstruction [%]  |    30    
Pixels in pupil          |  10284   
Field of view [arcsec]   |   0.00   
------------------------------------


------------- Source --------------
Source                   |   NGS   
Wavelength [m]           | 6.4e-07 
Zenith [arcsec]          |  0.00   
Azimuth [°]              |  0.00   
Altitude [m]             |   inf   
Magnitude                |  5.00   
Flux [photon/m²/s]       | 1.1e+08 
Coordinates [arcsec,deg] | [0,0]
-----------------------------------


------------- Source --------------
Source                   |   NGS   
Wavelength [m]           | 1.5e-06 
Zenith [arcsec]          |  0.00   
Azimuth [°]              |  0.00   
Altitude [m]             |   inf   
Magnitude                |  5.00   
Flux [photon/m²/s]       | 4.0e+07 
Coordinates [arcs

100%|██████████| 20/20 [00:01<00:00, 11.24it/s]


-------------Detector--------------
Sensor type              |   CCD   
Resolution [px]          |   240   
Gain                     |    1    
Quantum efficiency [%]   |   100   
Binning                  |   1x1   
Dark current [e-/px/s]   |  0.00   
Photon noise             |  False  
Bkg noise [e-]           |  False  
Readout noise [e-/px]    |   2.0   
-----------------------------------

-------------Detector--------------
Sensor type              |   CCD   
Resolution [px]          |   240   
Gain                     |    1    
Quantum efficiency [%]   |   100   
Binning                  |   1x1   
Dark current [e-/px/s]   |  0.00   
Photon noise             |  False  
Bkg noise [e-]           |  False  
Readout noise [e-/px]    |   0.0   
-----------------------------------



In [ ]:
wfs_frames = []
dm_commands = []
psf_frames = []
wfs_measurements = []
loop_status_list = []

wfs_timestamps = []
dm_timestamps = []
psf_timestamps = []

# These are the calibration data used to close the loop
calib_CL = calib_modal
M2C_CL = M2C_modal
reconstructor = M2C_CL@calib_CL.M 

t0 = time.time()

# initialize Telescope DM commands
dm.coefs=0
loop_status = 1

# You can update the the atmosphere parameter on the fly
atm.r0 = 0.12
atm.windSpeed = list(np.random.randint(1,10,atm.nLayer))
atm.windDirection = list(np.random.randint(0,360,atm.nLayer))

# To make sure to always replay the same turbulence, generate a new phase screen for the atmosphere and combine it with the Telescope
# atm.generateNewPhaseScreen(seed=12)

# combine telescope with atmosphere
tel+atm

# propagate both sources
ngs**atm*tel*ngs_cam
src**atm*tel*src_cam

# loop parameters
warmup = 20
nLoop = 5000  # number of iterations
gainCL = 0.4  # integrator gain
leakCL = 0.995 # integrator leak
wfs.cam.photonNoise = False  # enable photon noise on the WFS camera
display = True  # enable the display
frame_delay = 2  # number of frame delay
wfs_frame_step = 50  # keep one WFS frame in this many, like wfs_frame_step in telemetry.py
psf_frame_step = 20  # keep one WFS frame in this many, like wfs_frame_step in telemetry.py

# variables used to to save closed-loop data data
SR_ngs = np.zeros(nLoop+warmup)
SR_src = np.zeros(nLoop+warmup)

wfe_atmosphere = np.zeros(nLoop+warmup)
wfe_residual_SRC = np.zeros(nLoop+warmup)
wfe_residual_NGS = np.zeros(nLoop+warmup)
wfsSignal = np.arange(0, wfs.nSignal)*0  # buffer to simulate the loop delay

for i in range(nLoop + warmup):
    a = time.time()
    # update phase screens => overwrite tel.OPD and consequently tel.src.phase
    atm.update()
    # save the wave-front error of the incoming turbulence within the pupil
    wfe_atmosphere[i] = np.std(tel.OPD[np.where(tel.pupil > 0)])*1e9
    # propagate light from the ngs through the atmosphere, telescope, DM to the WFS and ngs camera
    ngs**atm*tel*dm*wfs*ngs_cam
    # propagate to the focal plane camera
    wfs*wfs.focal_plane_camera
    # save residuals corresponding to the ngs
    wfe_residual_NGS[i] = np.std(tel.OPD[np.where(tel.pupil > 0)])*1e9
    # save Strehl ratio from the PSF image
    SR_ngs[i] = strehlMeter(PSF=ngs_cam.frame, tel=tel, PSF_ref=ngs_psf_ref, display=False)
    # save the OPD seen by the ngs
    OPD_NGS = ngs.OPD.copy()
    if display:
        NGS_PSF = np.log10(np.abs(ngs_cam.frame))

    # propagate light from the src through the atmosphere, telescope, DM to the src camera
    src**atm*tel*dm*src_cam
    # save residuals corresponding to the SRC
    wfe_residual_SRC[i] = np.std(tel.OPD[np.where(tel.pupil > 0)])*1e9
    # save the OPD seen by the src
    OPD_SRC = src.OPD.copy()
    # save Strehl ratio from the PSF image
    SR_src[i] = strehlMeter(PSF=src_cam.frame, tel=tel, PSF_ref=src_psf_ref, display=False)

    # store the slopes after propagating to the WFS <=> 1 frames delay
    if frame_delay == 1:
        wfsSignal = wfs.signal

    # apply the commands on the DM
    dmResidual = np.matmul(reconstructor, wfsSignal)
    if loop_status:
        dm.coefs = leakCL * dm.coefs - gainCL*dmResidual
    else:
        dm.coefs = dm.coefs * 0

    # store the slopes after computing the commands <=> 2 frames delay
    if frame_delay == 2:
        wfsSignal = wfs.signal
    # print('Elapsed time: ' + str(time.time()-a) + ' s')

    # if i % 1000 == 0:
    #     loop_status = 1 - loop_status

    print('-----------------------------------')
    print('Loop'+str(i) + '/' + str(nLoop+warmup))
    print('NGS: Strehl ratio [%] : ', np.round(SR_ngs[i],1), ' WFE [nm] : ', np.round(wfe_residual_NGS[i],2))
    print('SRC: Strehl ratio [%] : ', np.round(SR_src[i],1), ' WFE [nm] : ', np.round(wfe_residual_SRC[i],2))

    if i > warmup:
        wfs_frames.append(wfs.cam.frame)
        dm_commands.append(dm.coefs)
        psf_frames.append(src_cam.frame)
        wfs_measurements.append(dmResidual)
        loop_status_list.append(loop_status)
        wfs_timestamps.append(t0 + tel.samplingTime*i)
        dm_timestamps.append(t0 + tel.samplingTime*i)
        psf_timestamps.append(t0 + tel.samplingTime*i)
    
#%% Closed Loop data analysis

plt.figure()
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, wfe_atmosphere, label='Turbulence')
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, wfe_residual_NGS, label='NGS')
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, wfe_residual_SRC, label='SRC')
plt.legend()
plt.xlabel('Time [s]')
plt.ylabel('WFE [nm]')

plt.figure()
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, SR_ngs, label='NGS@' + str(np.round(1e9*ngs.wavelength,0)) + ' nm')
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, SR_src, label='SRC@' + str(np.round(1e9*src.wavelength,0)) + ' nm')
plt.legend()
plt.xlabel('Time [s]')
plt.ylabel('SR [%]')

psf_frames = np.array(psf_frames[::psf_frame_step])
dm_commands = np.array(dm_commands)
wfs_frames = np.array(wfs_frames[::wfs_frame_step])
wfs_measurements = np.array(wfs_measurements)
loop_status_list = np.array(loop_status_list)
wfs_timestamps = np.array(wfs_timestamps[::wfs_frame_step])
dm_timestamps = np.array(dm_timestamps)
psf_timestamps = np.array(psf_timestamps[::psf_frame_step])


In [ ]:
wfs_frames = wfs_frames
psf_frames=psf_frames
dm_commands=dm_commands
m2c=M2C_CL
WFS_PUP=wfs.valid_signal_2D
loop_gain=gainCL
loop_leak=leakCL
wfs_fps=int(1/tel.samplingTime)
wfs_gain=1
sci_dit=src_cam.integrationTime
# one science frame is kept every psf_frame_step loop iterations
sci_fps=wfs_fps/psf_frame_step
sci_gain=1

if dm.nValidAct != config['dm']['n_actuators']:
    print(f"WARNING: the simulated DM has {dm.nValidAct} actuators, the instrument file says "
          f"{config['dm']['n_actuators']} (the file gets {dm.nValidAct} as Total_Number_Of_Actuators)")

In [ ]:
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

# Target attrs as telemetry.py writes them: SIMBAD values if the query works,
# otherwise only the name and a NaN elevation
target = "Dubhe"

location = EarthLocation(lat=config['site']['latitude_deg']*u.deg,
                         lon=config['site']['longitude_deg']*u.deg,
                         height=config['site']['height_m']*u.m)

star = None
try:
    Simbad.add_votable_fields('V', 'R', 'J', 'H')
    result = Simbad.query_object(target)
    if result is not None and len(result) > 0:
        row = result[0]
        star = {"main_id": str(row["main_id"]),
                "coord": SkyCoord(ra=row["ra"], dec=row["dec"], unit=(u.deg, u.deg), frame="icrs")}
        for band in ("V", "R", "J", "H"):
            star[band] = np.nan if np.ma.is_masked(row[band]) else float(row[band])
except Exception as e:
    print(f"SIMBAD query failed ({e})")

if star is not None:
    altaz = star["coord"].transform_to(AltAz(obstime=Time(psf_timestamps[0], format="unix"), location=location))
    elevation = altaz.alt.deg
    print(f"{target}: SIMBAD {star['main_id']}, V = {star['V']}")
    print(f"Elevation: {elevation:.2f} deg, azimuth: {altaz.az.deg:.2f} deg")
else:
    elevation = np.nan
    print(f"{target}: not found in SIMBAD, no magnitudes and NaN elevation")

In [ ]:
hdf5_dir.mkdir(parents=True, exist_ok=True)
file_name = hdf5_dir / f"simulated_data_r0m_{atm.r0:.2f}_V0mps_{atm.V0:.2f}_L0m_{atm.L0:.2f}_tau0ms_{atm.tau0*1e3:.2f}.hdf5"

with h5py.File(file_name, "w") as file:
    file.attrs["Instrument"] = config['instrument']['name']
    file.attrs["Telescope"] = config['instrument']['telescope']

    grp_wfs = file.create_group("WFS")
    # Loop status at every loop iteration (nonzero = closed loop), a dataset like
    # telemetry.py writes it: attributes are limited to 64 kB
    grp_wfs.create_dataset("loop_status", data=loop_status_list)
    grp_wfs.attrs["Loop_Gain"] = loop_gain
    grp_wfs.attrs["Loop_Leak"] = loop_leak
    grp_wfs.attrs["Loop_Freq"] = wfs_fps

    dset_wfs = grp_wfs.create_dataset("WFS_Images", data=wfs_frames)
    dset_wfs.attrs["Gain"] = wfs_gain
    dset_wfs.attrs["FPS"] = wfs_fps
    dset_wfs.attrs["Frame_Step"] = wfs_frame_step
    grp_wfs.create_dataset("WFS_TimeStamps", data=wfs_timestamps)
    grp_wfs.create_dataset("Dark", data=wfs_frames[0]*0)
    grp_wfs.create_dataset("Reference_Frame", data=wfs_frames[0]*0)

    grp_wfs.create_dataset("Valid_Pixel_Map", data=WFS_PUP)
    grp_wfs.create_dataset("DM_commands", data=dm_commands)
    grp_wfs.create_dataset("DM_TimeStamps", data=dm_timestamps)

    grp_wfs.create_dataset("DM_flat", data=dm_commands[0]*0)
    grp_wfs.create_dataset("DM_offset", data=dm_commands[0]*0)
    grp_wfs.create_dataset("DM_Map", data=dm.validAct)
    grp_wfs.create_dataset("WFS_measurements", data=wfs_measurements)

    grp_science = file.create_group("Science")
    grp_science.attrs["Target"] = target
    grp_science.attrs["Elevation"] = elevation
    if star is not None:
        grp_science.attrs["SIMBAD_ID"] = star["main_id"]
        grp_science.attrs["RA"] = star["coord"].ra.deg
        grp_science.attrs["Dec"] = star["coord"].dec.deg
        grp_science.attrs["Azimuth"] = altaz.az.deg
        grp_science.attrs["Vmag"] = star["V"]
        grp_science.attrs["Rmag"] = star["R"]
        grp_science.attrs["Jmag"] = star["J"]
        grp_science.attrs["Hmag"] = star["H"]
    grp_science.create_dataset("PSF_TimeStamps", data=psf_timestamps)

    dset_science = grp_science.create_dataset("Science_PSFs", data=psf_frames)
    dset_science.attrs["Exposure_Time"] = sci_dit
    dset_science.attrs["FPS"] = sci_fps
    dset_science.attrs["Gain"] = sci_gain
    dset_science.attrs["Sampling"] = config['science_camera']['sampling_at_calibration']
    dset_science.attrs["Wavelength"] = src.wavelength
    dset_science.attrs["Bandpass"] = config['science_camera']['bandpass_nm']*1e-9
    dset_science_dark = grp_science.create_dataset("Dark", data=psf_frames[0]*0)


    grp_calibration = file.create_group("Calibration")
    dset_iMat = grp_calibration.create_dataset("Interaction_Matrix", data=calib_CL.D)
    dset_iMat.attrs["Wavelength"] = ngs.wavelength
    grp_calibration.create_dataset("M2C", data=M2C_CL)
    grp_calibration.create_dataset("Z2C", data=M2C_zernike)
    grp_calibration.attrs["Diameter"] = config['telescope']['diameter_m']
    grp_calibration.attrs["Obstruction_ratio"] = config['telescope']['obstruction_ratio']
    grp_calibration.attrs["Science_Calibration_Wavelength"] = src.wavelength
    grp_calibration.attrs["AO_Calibration_Wavelength"] = ngs.wavelength
    grp_calibration.attrs["SkyCalibPupilRatio"] = config['dm']['sky_calib_pupil_ratio']
    grp_calibration.attrs["Actuators_in_diameter"] = config['dm']['actuators_in_diameter']
    grp_calibration.attrs["Total_Number_Of_Actuators"] = dm.nValidAct
    grp_calibration.attrs["Total_Number_Of_Controlled_Modes"] = M2C_CL.shape[1]
    grp_calibration.attrs["r0_reference_wvl"] = config['conventions']['r0_reference_wvl_nm']*1e-9

print(file_name)
